## Spark Session Setup & Data Loading

In [3]:
from pyspark.sql import SparkSession
# Create Spark Session
spark = SparkSession.builder.appName("RDD Lab").getOrCreate()
sc = spark.sparkContext

raw_rdd = sc.textFile("employees.txt")

raw_rdd.take(5)

['emp_id,name,department,job_title,salary,location,hire_date,performance_rating,years_exp',
 '1,John Smith,Engineering,Senior Developer,125000,San Francisco,2021-03-15,4.5,8',
 '2,Sarah Johnson,Sales,Account Executive,85000,New York,2022-01-10,4.2,5',
 '3,Michael Williams,Engineering,Software Engineer,95000,Austin,2023-06-20,3.8,3',
 '4,Jennifer Brown,Marketing,Marketing Manager,92000,Chicago,2020-11-05,4.7,7']

## Task 1 & 3 (Data Cleaning and Filtering)

In [ ]:
# Capture the header to exclude it from data
header = raw_rdd.first()

def parse_line(line):
    parts = line.split(",")

    # Basic validation: ensure the line has enough columns
    if len(parts) < 5:
        return None
    return parts

# Excluding header and empty lines and rows starting with "CORRUPT"
# Parsing strings into lists

clean_rdd = raw_rdd.filter(lambda x: x != header and x.strip() != "") \
                  .filter(lambda x: not x.startswith("CORRUPT")) \
                  .map(parse_line) \
                  .filter(lambda x: x is not None)

clean_rdd.collect()


[['1',
  'John Smith',
  'Engineering',
  'Senior Developer',
  '125000',
  'San Francisco',
  '2021-03-15',
  '4.5',
  '8'],
 ['2',
  'Sarah Johnson',
  'Sales',
  'Account Executive',
  '85000',
  'New York',
  '2022-01-10',
  '4.2',
  '5'],
 ['3',
  'Michael Williams',
  'Engineering',
  'Software Engineer',
  '95000',
  'Austin',
  '2023-06-20',
  '3.8',
  '3'],
 ['4',
  'Jennifer Brown',
  'Marketing',
  'Marketing Manager',
  '92000',
  'Chicago',
  '2020-11-05',
  '4.7',
  '7'],
 ['5',
  'David Jones',
  'Finance',
  'Senior Analyst',
  '105000',
  'Boston',
  '2021-08-12',
  '4.3',
  '6'],
 ['6',
  'Lisa Garcia',
  'IT',
  'DevOps Engineer',
  '115000',
  'Seattle',
  '2022-04-18',
  '4.6',
  '4'],
 ['7',
  'Robert Martinez',
  'Legal',
  'Legal Counsel145000',
  'San Francisco',
  '2019-09-22',
  '4.8',
  '10',
  ' 5'],
 ['8',
  'Patricia Wilson',
  'HR',
  'HR Manager',
  '88000',
  'Denver',
  '2020-02-28',
  '4.1',
  '6'],
 ['9',
  'James Anderson',
  'Sales',
  'Sales Mana

## Task 2 (Counting Name Occurrences)

In [5]:
# Map each name to a count of 1, then reduce by name
name_counts = clean_rdd.map(lambda x: (x[1], 1)) \
                      .reduceByKey(lambda a, b: a + b)
name_counts.collect()

[('Sarah Johnson', 1),
 ('Michael Williams', 1),
 ('Jennifer Brown', 1),
 ('David Jones', 1),
 ('Lisa Garcia', 1),
 ('Patricia Wilson', 1),
 ('James Anderson', 1),
 ('Mary Thomas', 1),
 ('John Smith', 1),
 ('Robert Martinez', 1)]

 ## Task 4 (Average Salary per Department)

In [ ]:
def get_salary_info(parts):
    try:
        dept = parts[2]
        # Attempt to convert salary to float (handles simple dirty data)
        salary = float(parts[4])
        return (dept, (salary, 1))
    except:
        # If conversion fails, return 0 for that record
        return (parts[2], (0.0, 0))

# Calculation sum of salaries and count of employees per dept, then divide
avg_salary = clean_rdd.map(get_salary_info) \
                     .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])) \
                     .mapValues(lambda v: v[0] / v[1] if v[1] > 0 else 0)

avg_salary.collect()


[('Engineering', 121666.66666666667),
 ('Sales', 97500.0),
 ('Finance', 105000.0),
 ('IT', 115000.0),
 ('Legal', 0),
 ('HR', 88000.0),
 ('Marketing', 92000.0)]

## Task 5 (Employee Count per Department)


In [7]:
# Map each department to 1, then sum them up
dept_counts = clean_rdd.map(lambda x: (x[2], 1)) \
                      .reduceByKey(lambda a, b: a + b)

dept_counts.collect()


[('Engineering', 3),
 ('Sales', 2),
 ('Finance', 1),
 ('IT', 1),
 ('Legal', 1),
 ('HR', 1),
 ('Marketing', 1)]